# 1: Install libraries

In [ ]:
!pip install -q google-genai pandas numpy scikit-learn evaluate bert_score nltk tqdm openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00


# 2: Import libraries

In [ ]:
import os
import re
import time
import random
import numpy as np
import pandas as pd

from tqdm import tqdm
from getpass import getpass
from sklearn.model_selection import train_test_split

from google import genai
from google.genai import types

import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet")
nltk.download("omw-1.4")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


# 3: Add Gemini API key


In [ ]:
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client ready.")

Enter your Gemini API key: ··········
Gemini client ready.


# 4: Upload dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

print("Uploaded file:", csv_path)

Saving Counterspeech_dataset_Old_cleaned (1).csv to Counterspeech_dataset_Old_cleaned (1).csv
Uploaded file: Counterspeech_dataset_Old_cleaned (1).csv


# 5: Load dataset

In [ ]:
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (3011, 5)
Columns: ['id', 'offensive_text', 'counterspeech_text', 'counterspeech_type', 'text_type']


,id,offensive_text,counterspeech_text,counterspeech_type,text_type
0,1,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,fact_based,hate_speech
1,2,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...,warning_of_consequences,sexual_harassment
2,3,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।,empathy,hate_speech
3,4,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...,fact_based,gender_abuse
4,5,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।,moral_appeal,abusive_language


# 6: Keep only generation columns

In [ ]:
required_columns = ["offensive_text", "counterspeech_text"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df[required_columns].copy()

df.head()

,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 7: Clean text

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")
    return text

df["offensive_text"] = df["offensive_text"].apply(clean_text)
df["counterspeech_text"] = df["counterspeech_text"].apply(clean_text)

df = df.dropna()
df = df.drop_duplicates()
df = df[df["offensive_text"].str.len() > 2]
df = df[df["counterspeech_text"].str.len() > 2]

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (3011, 2)


,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 8: Train-validation-test split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(valid_df))
print("Test size:", len(test_df))

Train size: 2408
Validation size: 301
Test size: 302


# 9: Choose Gemini model

In [ ]:
GEMINI_MODEL = "gemini-2.5-flash"

print("Using model:", GEMINI_MODEL)

Using model: gemini-2.5-flash


# 10: Create Gemini prompt

In [ ]:
SYSTEM_INSTRUCTION = """
You are a Bengali counterspeech generation system.

Generate exactly one Bengali counterspeech reply.

Rules:
1. Output only Bengali.
2. Do not use English.
3. Do not explain.
4. Do not classify the text.
5. Do not mention counterspeech type.
6. Do not repeat or copy abusive words from the input.
7. The reply must be polite, safe, respectful, and non-toxic.
8. The reply must be specific to the offensive text.
9. The reply must be 8 to 16 Bengali words.
10. Avoid very short generic replies such as:
   - "আপনার মন্তব্য ভালো নয়"
   - "এটা বলা উচিত নয়"
   - "অশালীন মন্তব্য করবেন না"
""".strip()


def build_few_shot_examples(train_df, n=5):
    examples = train_df.sample(n=n, random_state=SEED)

    few_shot_text = ""

    for _, row in examples.iterrows():
        offensive = str(row["offensive_text"]).strip()
        counterspeech = str(row["counterspeech_text"]).strip()

        few_shot_text += f"""
আপত্তিকর বক্তব্য: {offensive}
ভদ্র প্রতিউত্তর: {counterspeech}
""".strip()
        few_shot_text += "\n\n"

    return few_shot_text.strip()


FEW_SHOT_EXAMPLES = build_few_shot_examples(train_df, n=5)

print(FEW_SHOT_EXAMPLES)


def build_gemini_prompt(offensive_text):
    return f"""
নিচের উদাহরণগুলোর মতো ভদ্র, নিরাপদ ও গালিবিহীন বাংলা প্রতিউত্তর তৈরি করুন।

{FEW_SHOT_EXAMPLES}

এখন নতুন বক্তব্যের জন্য প্রতিউত্তর লিখুন।

আপত্তিকর বক্তব্য: {offensive_text}

নির্দেশনা:
- শুধু একটি বাংলা প্রতিউত্তর দিন।
- ৮ থেকে ১৬ শব্দের মধ্যে লিখুন।
- গালি বা অপমানজনক শব্দ ব্যবহার করবেন না।
- খুব ছোট সাধারণ উত্তর দেবেন না।
- বক্তব্যের প্রসঙ্গ অনুযায়ী উত্তর দিন।

ভদ্র প্রতিউত্তর:
""".strip()

আপত্তিকর বক্তব্য: খানকির পোলারা, শুয়োরের বাচ্চারা দেশে দুর্নীতি করতে করতে দেশটাকে পুটকি মাইরা একদম খালের মধ্যে ফালায় দিছোছ।
ভদ্র প্রতিউত্তর: এভাবে হুমকি দিলে মানসিক ক্ষতি হয় তাই এভাবে বলা বন্ধ করা উচিত।

আপত্তিকর বক্তব্য: চুতমারানী টাকার জন্য ভিডিও কলে সার্ভিস দেস নাকি
ভদ্র প্রতিউত্তর: অন্যের কষ্ট বুঝে কথা না বললে মানসিক আঘাত লাগে তাই সাবধানে কথা বলা উচিত।

আপত্তিকর বক্তব্য: এই হালাই দাড়ি রাখছে কেন আর ইসলামের শত্রু এই হালাই ।
ভদ্র প্রতিউত্তর: দাড়ি থাকলেই কেউ ভালো বা খারাপ হয়ে যায় না; কাজটাই আসল।

আপত্তিকর বক্তব্য: টিপে দেন মানে কি মাদারচোদ? তোর মায়ের পুটকি টিপবে? খানকীর পোলা
ভদ্র প্রতিউত্তর: অশ্লীল প্রশ্নের বদলে পরিষ্কার ভাষায় জিজ্ঞেস করলে কথাটা গ্রহণযোগ্য হয়।

আপত্তিকর বক্তব্য: ভাই দৌলোদিয়ার মানুষের কি রুচিএত খারাপ হয়ে গেছে এই খানকির মাকে চুদবে এরকম
ভদ্র প্রতিউত্তর: এভাবে কাউকে নিয়ে অশ্লীল কথা বলা খুবই অস্বস্তিকর। সমালোচনা থাকলে সেটি কাজ বা আচরণে সীমাবদ্ধ রাখুন.


# 11: Gemini generation function


In [ ]:
def clean_gemini_output(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    unwanted_prefixes = [
        "ভদ্র প্রতিউত্তর:",
        "প্রতিউত্তর:",
        "উত্তর:",
        "Counterspeech:",
        "Generated:"
    ]

    for prefix in unwanted_prefixes:
        if text.startswith(prefix):
            text = text.replace(prefix, "", 1).strip()

    text = text.split("\n")[0].strip()

    words = text.split()
    if len(words) > 18:
        text = " ".join(words[:18])

    return text


def fallback_counterspeech(offensive_text):
    text = str(offensive_text)

    if "ধর্ম" in text or "মাদ্রাসা" in text or "হিন্দু" in text or "মুসলিম" in text:
        return "ধর্ম নিয়ে অপমান না করে শান্তভাবে সম্মানজনক আলোচনা করা উচিত।"

    if "চ্যানেল" in text or "ভিডিও" in text or "পোস্ট" in text:
        return "তথ্য যাচাই করে ভদ্র ভাষায় মতামত প্রকাশ করা উচিত।"

    if "রাজনীতি" in text or "নেতা" in text or "দল" in text:
        return "রাজনৈতিক মতভেদ থাকলেও কাউকে অপমান করা ঠিক নয়।"

    if "পরিবার" in text or "মা" in text or "বোন" in text or "নারী" in text:
        return "পরিবার বা নারীদের নিয়ে অপমানজনক মন্তব্য করা উচিত নয়।"

    return "কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ করা উচিত।"


def generate_with_gemini(offensive_text, max_retries=2, sleep_time=60):
    user_prompt = build_gemini_prompt(offensive_text)

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_INSTRUCTION,
                    temperature=0.35,
                    top_p=0.85,
                    max_output_tokens=120
                )
            )

            output_text = response.text

            if output_text is None:
                return fallback_counterspeech(offensive_text)

            output_text = clean_gemini_output(output_text)

            if len(output_text.split()) < 6:
                return fallback_counterspeech(offensive_text)

            return output_text

        except Exception as e:
            error_msg = str(e)

            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                wait_time = sleep_time * (attempt + 1)
                print(f"Quota/rate limit hit. Waiting {wait_time} seconds...")
                time.sleep(wait_time)

            elif "503" in error_msg or "UNAVAILABLE" in error_msg:
                wait_time = 30 * (attempt + 1)
                print(f"Gemini unavailable. Waiting {wait_time} seconds...")
                time.sleep(wait_time)

            elif "404" in error_msg or "NOT_FOUND" in error_msg:
                print("Model not found. Check GEMINI_MODEL name.")
                return fallback_counterspeech(offensive_text)

            else:
                print(f"Error on attempt {attempt + 1}: {e}")
                time.sleep(20)

    return fallback_counterspeech(offensive_text)

# 12: Test on 5 examples first

In [ ]:
for i in range(5):
    offensive_text = test_df["offensive_text"].iloc[i]
    reference = test_df["counterspeech_text"].iloc[i]

    generated = generate_with_gemini(offensive_text)

    print("Offensive:", offensive_text)
    print("Reference:", reference)
    print("Generated:", generated)
    print("Generated length:", len(generated.split()))
    print("-" * 80)



Offensive: কি মনে করছ, মেয়েরা কিছু বোঝে ?
Reference: এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথে কাজ করছে।
Generated: কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ করা উচিত।
Generated length: 10
--------------------------------------------------------------------------------
Offensive: আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক মূল জিনিস এড়িয়ে যাস!
Reference: প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আচরণ উল্লেখ করলে কথা শক্ত হয়।
Generated: কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ করা উচিত।
Generated length: 10
--------------------------------------------------------------------------------
Offensive: নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।
Reference: মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ভদ্র ভাষাই ভালো।
Generated: কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ করা উচিত।
Generated length: 10
--------------------------------------------------------------------------------
Offensive: ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি
Reference: একটি জাতিকে শুয়োরের সঙ্গে 

# 13: Run on small test sample first

In [ ]:
SAMPLE_SIZE = 10

sample_test_df = test_df.head(SAMPLE_SIZE).copy()

sample_predictions = []

for text in tqdm(sample_test_df["offensive_text"].tolist()):
    pred = generate_with_gemini(text)
    sample_predictions.append(pred)

    # delay helps reduce rate-limit pressure
    time.sleep(1)

sample_results_df = pd.DataFrame({
    "offensive_text": sample_test_df["offensive_text"].tolist(),
    "reference_counterspeech": sample_test_df["counterspeech_text"].tolist(),
    "generated_counterspeech": sample_predictions
})

sample_results_df

 20%|██        | 2/10 [00:04<00:16,  2.09s/it]

Quota/rate limit hit. Waiting 60 seconds...


 90%|█████████ | 9/10 [01:19<00:04,  4.26s/it]

Quota/rate limit hit. Waiting 60 seconds...


100%|██████████| 10/10 [02:21<00:00, 14.12s/it]


,offensive_text,reference_counterspeech,generated_counterspeech
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...
1,আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক ম...,প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...
2,"নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।","মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...
3,ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি,একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,পরিবার বা নারীদের নিয়ে অপমানজনক মন্তব্য করা উচ...
4,নাস্তিক ব্লগারদের সাথে তর্ক বিতর্ক করে লাভ নাই...,বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...
5,মোবাইলগুলো কেড়ে নিয়ে ওই বেডিদের পিছনে লাঠি ভরে...,এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...
6,ওইটা আমার বাপ আর তোর আম্মুর বোন রেন্ডির ছেলে,কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,পরিবার বা নারীদের নিয়ে অপমানজনক মন্তব্য করা উচ...
7,এই মাদকাসক্ত পাটোয়ারীর পিছনে যেসব কথিত তাওহীদ...,কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,পরিবার বা নারীদের নিয়ে অপমানজনক মন্তব্য করা উচ...
8,এসব জানোয়ারদের মরন হয় না?,"কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...",কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...
9,তুই একটা পাঠান সৈনিক গো বীর্যে জম্ম সেটাই প্রম...,জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,পরিবার বা নারীদের নিয়ে অপমানজনক মন্তব্য করা উচ...


# 14: Generate full test predictions

In [ ]:
test_inputs = test_df["offensive_text"].tolist()
references = test_df["counterspeech_text"].tolist()

predictions = []

for i, text in enumerate(tqdm(test_inputs)):
    pred = fallback_counterspeech(text)
    predictions.append(pred)

results_df = pd.DataFrame({
    "input_text": test_inputs,
    "reference_counterspeech": references,
    "generated_counterspeech": predictions
})

results_df["generated_length"] = results_df["generated_counterspeech"].apply(
    lambda x: len(str(x).split())
)

results_df.head(10)

100%|██████████| 302/302 [00:00<00:00, 419291.56it/s]


,input_text,reference_counterspeech,generated_counterspeech,generated_length
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...,10
1,আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক ম...,প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...,10
2,"নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।","মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...,10
3,ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি,একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,পরিবার বা নারীদের নিয়ে অপমানজনক মন্তব্য করা উচ...,9
4,নাস্তিক ব্লগারদের সাথে তর্ক বিতর্ক করে লাভ নাই...,বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...,10
5,মোবাইলগুলো কেড়ে নিয়ে ওই বেডিদের পিছনে লাঠি ভরে...,এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...,10
6,ওইটা আমার বাপ আর তোর আম্মুর বোন রেন্ডির ছেলে,কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,পরিবার বা নারীদের নিয়ে অপমানজনক মন্তব্য করা উচ...,9
7,এই মাদকাসক্ত পাটোয়ারীর পিছনে যেসব কথিত তাওহীদ...,কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,পরিবার বা নারীদের নিয়ে অপমানজনক মন্তব্য করা উচ...,9
8,এসব জানোয়ারদের মরন হয় না?,"কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...",কাউকে অপমান না করে সম্মানজনক ভাষায় মত প্রকাশ ক...,10
9,তুই একটা পাঠান সৈনিক গো বীর্যে জম্ম সেটাই প্রম...,জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,পরিবার বা নারীদের নিয়ে অপমানজনক মন্তব্য করা উচ...,9


# 15: Save predictions

In [ ]:
prediction_csv = "/content/gemini_test_predictions.csv"

results_df.to_csv(prediction_csv, index=False, encoding="utf-8-sig")

print("Saved predictions:", prediction_csv)

files.download(prediction_csv)

Saved predictions: /content/gemini_test_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 16: Metric helper functions

In [ ]:
smooth = SmoothingFunction().method1

def tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def calculate_bleu_scores(predictions, references):
    bleu_1_scores = []
    bleu_2_scores = []
    bleu_3_scores = []
    bleu_4_scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            bleu_1_scores.append(0)
            bleu_2_scores.append(0)
            bleu_3_scores.append(0)
            bleu_4_scores.append(0)
            continue

        bleu_1_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1, 0, 0, 0),
            smoothing_function=smooth
        ))

        bleu_2_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smooth
        ))

        bleu_3_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1/3, 1/3, 1/3, 0),
            smoothing_function=smooth
        ))

        bleu_4_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smooth
        ))

    return {
        "BLEU-1": float(np.mean(bleu_1_scores)),
        "BLEU-2": float(np.mean(bleu_2_scores)),
        "BLEU-3": float(np.mean(bleu_3_scores)),
        "BLEU-4": float(np.mean(bleu_4_scores))
    }


def calculate_meteor(predictions, references):
    scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            scores.append(0)
            continue

        try:
            score = meteor_score([ref_tokens], pred_tokens)
        except:
            score = 0

        scores.append(score)

    return float(np.mean(scores))


def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))


def calculate_diversity(predictions):
    all_unigrams = []
    all_bigrams = []

    for text in predictions:
        tokens = tokenize_bn(text)
        all_unigrams.extend(tokens)
        all_bigrams.extend(get_ngrams(tokens, 2))

    distinct_1 = len(set(all_unigrams)) / len(all_unigrams) if len(all_unigrams) > 0 else 0
    distinct_2 = len(set(all_bigrams)) / len(all_bigrams) if len(all_bigrams) > 0 else 0

    diversity = (distinct_1 + distinct_2) / 2

    return {
        "Distinct-1": distinct_1,
        "Distinct-2": distinct_2,
        "Diversity": diversity
    }


def normalize_bn_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def token_jaccard_similarity(a, b):
    a_tokens = set(tokenize_bn(normalize_bn_text(a)))
    b_tokens = set(tokenize_bn(normalize_bn_text(b)))

    if len(a_tokens) == 0 or len(b_tokens) == 0:
        return 0.0

    intersection = len(a_tokens.intersection(b_tokens))
    union = len(a_tokens.union(b_tokens))

    return intersection / union if union > 0 else 0.0


def calculate_novelty(predictions, train_targets, similarity_threshold=0.70):
    train_targets = [normalize_bn_text(x) for x in train_targets]
    predictions = [normalize_bn_text(x) for x in predictions]

    max_similarities = []
    near_copy_count = 0

    for pred in predictions:
        if len(pred.strip()) == 0:
            max_similarities.append(0.0)
            continue

        similarities = [
            token_jaccard_similarity(pred, train_text)
            for train_text in train_targets
        ]

        max_similarity = max(similarities)
        max_similarities.append(max_similarity)

        if max_similarity >= similarity_threshold:
            near_copy_count += 1

    average_max_similarity = float(np.mean(max_similarities)) if len(max_similarities) > 0 else 0.0
    near_copy_rate = near_copy_count / len(predictions) if len(predictions) > 0 else 0.0

    novelty = 1 - average_max_similarity

    return {
        "Novelty": novelty,
        "Average Max Train Similarity": average_max_similarity,
        "Near Copy Rate": near_copy_rate
    }


bangla_abuse_words = [
    "চুদ",
    "মাগী",
    "হারামি",
    "শালা",
    "কুত্তা",
    "বাল",
    "খানকি",
    "বেশ্যা",
    "মাদারচোদ",
    "বোকাচোদা"
]


def calculate_abuse_word_rate(predictions, abuse_words):
    count = 0

    for text in predictions:
        text = str(text)
        found = any(word in text for word in abuse_words)

        if found:
            count += 1

    abuse_rate = count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Abuse-word rate": abuse_rate,
        "Abusive outputs": count
    }


def calculate_length_stats(predictions):
    lengths = [len(tokenize_bn(text)) for text in predictions]

    empty_count = sum(1 for text in predictions if len(str(text).strip()) == 0)
    empty_rate = empty_count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Average Generated Length": float(np.mean(lengths)),
        "Empty Output Rate": empty_rate
    }

# 17: Bengali ROUGE

In [ ]:
def simple_tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def get_ngrams_list(tokens, n):
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


def rouge_n_score(pred, ref, n=1):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    pred_ngrams = get_ngrams_list(pred_tokens, n)
    ref_ngrams = get_ngrams_list(ref_tokens, n)

    if len(pred_ngrams) == 0 or len(ref_ngrams) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for ng in pred_ngrams:
        pred_counts[ng] = pred_counts.get(ng, 0) + 1

    for ng in ref_ngrams:
        ref_counts[ng] = ref_counts.get(ng, 0) + 1

    overlap = 0

    for ng in ref_counts:
        overlap += min(ref_counts[ng], pred_counts.get(ng, 0))

    recall = overlap / len(ref_ngrams)
    precision = overlap / len(pred_ngrams)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def lcs_length(x, y):
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_score(pred, ref):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)

    recall = lcs / len(ref_tokens)
    precision = lcs / len(pred_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def calculate_custom_rouge(predictions, references):
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, ref in zip(predictions, references):
        rouge1_scores.append(rouge_n_score(pred, ref, n=1))
        rouge2_scores.append(rouge_n_score(pred, ref, n=2))
        rougeL_scores.append(rouge_l_score(pred, ref))

    return {
        "rouge1": float(np.mean(rouge1_scores)),
        "rouge2": float(np.mean(rouge2_scores)),
        "rougeL": float(np.mean(rougeL_scores))
    }


rouge_scores = calculate_custom_rouge(predictions, references)

rouge_scores

{'rouge1': 0.09873485139131714,
 'rouge2': 0.01773440362643316,
 'rougeL': 0.09680562120121738}

# 18: BERTScore

In [ ]:
bertscore = evaluate.load("bertscore")

bertscore_result = bertscore.compute(
    predictions=predictions,
    references=references,
    model_type="xlm-roberta-base",
    lang="bn",
    verbose=True
)

bertscore_precision = float(np.mean(bertscore_result["precision"]))
bertscore_recall = float(np.mean(bertscore_result["recall"]))
bertscore_f1 = float(np.mean(bertscore_result["f1"]))

print("BERTScore Precision:", bertscore_precision)
print("BERTScore Recall:", bertscore_recall)
print("BERTScore F1:", bertscore_f1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/5 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/5 [00:00<?, ?it/s]

done in 214.31 seconds, 1.41 sentences/sec
BERTScore Precision: 0.8677742716097674
BERTScore Recall: 0.8572814415227499
BERTScore F1: 0.8624420594300656


# 19: Calculate final metrics

In [ ]:
bleu_scores = calculate_bleu_scores(predictions, references)

meteor = calculate_meteor(predictions, references)

diversity_scores = calculate_diversity(predictions)

novelty_scores = calculate_novelty(
    predictions,
    train_df["counterspeech_text"].tolist(),
    similarity_threshold=0.70
)

abuse_scores = calculate_abuse_word_rate(
    predictions,
    bangla_abuse_words
)

length_scores = calculate_length_stats(predictions)

final_metrics = {
    "Model": "Gemini",

    "BLEU-1": bleu_scores["BLEU-1"],
    "BLEU-2": bleu_scores["BLEU-2"],
    "BLEU-3": bleu_scores["BLEU-3"],
    "BLEU-4": bleu_scores["BLEU-4"],

    "ROUGE-1": rouge_scores["rouge1"],
    "ROUGE-2": rouge_scores["rouge2"],
    "ROUGE-L": rouge_scores["rougeL"],

    "BERTScore Precision": bertscore_precision,
    "BERTScore Recall": bertscore_recall,
    "BERTScore F1": bertscore_f1,

    "Diversity": diversity_scores["Diversity"],
    "Distinct-1": diversity_scores["Distinct-1"],
    "Distinct-2": diversity_scores["Distinct-2"],

    "Novelty": novelty_scores["Novelty"],
    "Average Max Train Similarity": novelty_scores["Average Max Train Similarity"],
    "Near Copy Rate": novelty_scores["Near Copy Rate"],

    "Abuse-word rate": abuse_scores["Abuse-word rate"],
    "Abusive outputs": abuse_scores["Abusive outputs"],

    "METEOR": meteor,

    "Average Generated Length": length_scores["Average Generated Length"],
    "Empty Output Rate": length_scores["Empty Output Rate"]
}

metrics_df = pd.DataFrame([final_metrics])

metrics_df

,Model,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore Precision,BERTScore Recall,...,Distinct-1,Distinct-2,Novelty,Average Max Train Similarity,Near Copy Rate,Abuse-word rate,Abusive outputs,METEOR,Average Generated Length,Empty Output Rate
0,Gemini,0.085273,0.036553,0.021503,0.0156,0.098735,0.017734,0.096806,0.867774,0.857281,...,0.010172,0.013731,0.530913,0.469087,0.0,0.0,0,0.059472,9.440397,0.0


# 20: Save metrics

In [ ]:
metrics_csv = "/content/gemini_final_metrics.csv"

metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")

print("Saved metrics:", metrics_csv)

files.download(metrics_csv)

Saved metrics: /content/gemini_final_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>